<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Regional Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **Optional - Switch outputs to S3 / MinIO**

Flip `USE_S3 = True` below to route every **write** (results, partials,
`failed_ids` CSV, HTML report) to S3-compatible storage instead of local
disk. Reads in Step 2 (entity loading from Geosys' S3 via
`EDA_S3_cloudstore`) are unaffected — they use the WorkflowManager's
pre-instantiated boto3 client and are immune to env-var changes.

Two presets:

- **MinIO (local dev)**: `docker compose -f tools/minio-compose.yml up -d`,
  leave the defaults below as-is. The bucket is auto-created if missing.
- **Real AWS S3**: set `S3_ENDPOINT = None` to drop the endpoint override
  and rely on your existing AWS credentials (`.env` / IAM role /
  instance profile). Provide your bucket name in `S3_BUCKET`.

Leave `USE_S3 = False` to keep the existing local-disk behaviour.
Requires `pip install -e ".[s3]"` for `s3fs`.


In [ ]:
import os
import uuid

# --------------------------------------------------------------------------
# Toggle
# --------------------------------------------------------------------------
USE_S3 = False                              # flip to True to route writes to S3
S3_BUCKET = "earthdaily-agriculture-dev"                    # bucket name (auto-created on MinIO)
S3_ENDPOINT = "http://localhost:9000"       # MinIO default; None for real AWS
S3_RUN_PREFIX = None                        # None -> auto-generate per kernel boot

if USE_S3:
    if S3_ENDPOINT:
        # MinIO presets — overrides any real AWS creds in the environment.
        os.environ.setdefault("AWS_ACCESS_KEY_ID", "minioadmin")
        os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minioadmin")
        os.environ["AWS_ENDPOINT_URL"] = S3_ENDPOINT
        os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
        # Pre-create the bucket if it doesn't exist (MinIO requires explicit mkdir).
        try:
            import s3fs
        except ImportError:
            raise ImportError('s3fs not installed - run: pip install -e ".[s3]"')
        fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})
        if not fs.exists(S3_BUCKET):
            fs.mkdir(S3_BUCKET)
            print(f"  Created bucket: s3://{S3_BUCKET}")
    else:
        # Real AWS path — rely on the existing boto3 default credential chain.
        pass

    if S3_RUN_PREFIX is None:
        S3_RUN_PREFIX = f"runs/regional/{uuid.uuid4().hex[:8]}"

    # Swap every write target to S3. The cache_dir gate inside BaseExtractor
    # auto-disables caching when cache_dir is a remote URI (with a warning).
    base = f"s3://{S3_BUCKET}/{S3_RUN_PREFIX}"
    manager.config["output_result_dir"]  = f"{base}/results"
    manager.config["partial_result_dir"] = f"{base}/partials"
    manager.config["cache_dir"]          = f"{base}/cache"
    manager.output_result_dir  = manager.config["output_result_dir"]
    manager.partial_result_dir = manager.config["partial_result_dir"]

    print(f"S3 writes enabled - base: {base}/")
    print(f"  results : {manager.config['output_result_dir']}")
    print(f"  partials: {manager.config['partial_result_dir']}")
    print(f"  cache   : {manager.config['cache_dir']}  (auto-disabled, see warning above)")
else:
    print("Local mode - writes go to project-root results/, partials/, etc.")


## **🛠️ Step 2: Get entities**

In [ ]:
from earthdaily.agriculture.services.S3 import EDA_S3_cloudstore
amu = 'GIS_data/Shapefiles/AQ_blocks/Block_281_DEUGermanyDistricts_AMUS.csv'
bucket_name = 'key-account-storage'
amu_df = EDA_S3_cloudstore.read_from_s3(
    manager.authenticator, 
    bucket_name, 
    amu, 
    verbose=True,
    on_bad_lines='warn',
    sep=';',  
    quoting=1,  # Handle quoted fields properly
    encoding='utf-8'
)

## **📥 Step 3: Extract analytics - Debug function from regional_ts_functions.py**

### 🗺️ Configure extraction

In [ ]:
from earthdaily.agriculture.extractors.regional_ts_extractor import RegionalExtractor
extractor = RegionalExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

extractor.setup_regional_parameters(
        index="vegetation-vigor-index",
        start_date="2018-01-01",
        # end_date="2026-12-31",
        fillyeargap=False,
        idblock=281,
        idpixeltype=1,
        indicatorTypeIds=[1]
)

### 🗺️ Test functions

#### API call

In [ ]:
result = extractor.get_regional_ts_by_id({"amu_id": 2432528})


In [ ]:
# amu with error 
result = extractor.get_regional_ts_by_id({"amu_id": 2121566})

#### Test get_regional_ts_safe

In [ ]:
print("\n--- Test: get_regional_ts_safe ---")
safe_result = extractor.get_regional_ts_by_id_safe({"amu_id": 2121564})
print(safe_result)

#### Test format_regional_json

In [ ]:
print("\n--- Test: format_regional_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df, formatted_df_avg = extractor.format_regional_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
    display(formatted_df_avg.head())
else:
    print("⚠️ Skipping format_regional_json: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
# Test data
import pandas as pd
row = pd.Series({
    "amu_id": 2432528,
})
# Extractor request
result = extractor.process_single_entity_regional(row)

In [ ]:
#print(result["observed_data"])
print(result["daily_avg_data"])

### 🗺️ process_inseason_bulk_extraction_parallel

In [ ]:
print(amu_df.columns)

In [ ]:
first_row = amu_df.iloc[0]
result = extractor.process_single_entity_regional(first_row)
print("Result keys:", result.keys())
print("\nObserved data:")
print("  Type:", type(result.get("observed_data")))
print("  Is None:", result.get("observed_data") is None)
if result.get("observed_data") is not None:
    print("  Shape:", result.get("observed_data").shape)
    print("  Empty:", result.get("observed_data").empty)
    print("  Columns:", result.get("observed_data").columns.tolist())

print("\nDaily avg data:")
print("  Type:", type(result.get("daily_avg_data")))
print("  Is None:", result.get("daily_avg_data") is None)
if result.get("daily_avg_data") is not None:
    print("  Shape:", result.get("daily_avg_data").shape)
    print("  Empty:", result.get("daily_avg_data").empty)
    print("  Columns:", result.get("daily_avg_data").columns.tolist())

print("\nError:", result.get("error"))

In [ ]:
print(first_row)

In [ ]:
top25 = amu_df.head(20)
#add specific logger for this extraction
# extractor.logger.remove()
# extractor.logger.add(
#     lambda msg: print(msg, end=""),
#     level="WARNING" #(TRACE, DEBUG, INFO, SUCCESS, WARNING, ERROR, CRITICAL)
#)

results = extractor.process_entity_regional_bulk_parallel(
    entity_list=top25,
    params=None,          # or pass overrides here
    max_workers=2,        # adjust threads depending on API rate limits !!!! this is very low for AQ APIs
    output_path=manager.output_result_dir,
    partial_frequency= 50,
    fail_safe= False,
    # filter_column="",
    # filter_value="",
    # filter_type="include", # filter type used to 'include' or 'exclude' row matching column and value filter
    merge_existing='auto',
    skip_export=False,
    prefix='regional'
)



## **🧪 Step 4: Test dynamic column mapping for `amu_id`**

Test that `column_mapping={"amu_id": "<custom_col>"}` works across all methods:
- Single API call (`get_regional_ts_by_id`)
- Safe wrapper (`get_regional_ts_by_id_safe`)
- Single entity processing (`process_single_entity_regional`)
- Bulk parallel extraction (`process_entity_regional_bulk_parallel`)

### 🔄 Prepare test data with renamed column

Simulate user DataFrame where the AMU ID column has a custom name (`region_code`) instead of the default `amu_id`.

In [ ]:
# Create a test DataFrame with a custom column name instead of 'amu_id'
amu_df_custom = amu_df.copy()
amu_df_custom = amu_df_custom.rename(columns={"amu_id": "region_code"})
print("Original columns:", amu_df.columns.tolist())
print("Custom columns:  ", amu_df_custom.columns.tolist())
print(f"\nFirst 3 region_code values: {amu_df_custom['region_code'].head(3).tolist()}")

### 🛠️ Configure extractor with column mapping

In [ ]:
# Create a new extractor instance with column mapping
from earthdaily.agriculture.extractors.regional_ts_extractor import RegionalExtractor

extractor_mapped = RegionalExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

# Pass column_mapping to tell the extractor that 'amu_id' is called 'region_code' in our DataFrame
extractor_mapped.setup_regional_parameters(
    index="vegetation-vigor-index",
    start_date="2023-06-06",
    end_date="2025-06-15",
    fillyeargap=False,
    idblock=141,
    idpixeltype=8,
    indicatorTypeIds=[1],
    column_mapping={"amu_id": "region_code"}  # <-- map amu_id to custom column name
)

# Verify the mapping is set
print(f"\nColumn mapping for 'amu_id': {extractor_mapped.get_mapped_column('amu_id')}")

### 🧪 Test API call with mapped column

In [ ]:
# Test get_regional_ts_by_id with custom column name ('region_code' instead of 'amu_id')
print("--- Test: get_regional_ts_by_id with column_mapping ---")
result_mapped = extractor_mapped.get_regional_ts_by_id({"region_code": 2121564})
print(f"✅ API call succeeded with mapped column. Got {len(result_mapped.get('observedMeasures', []))} observed measures.")

### 🧪 Test safe wrapper with mapped column

In [ ]:
# Test get_regional_ts_by_id_safe with custom column name
print("--- Test: get_regional_ts_by_id_safe with column_mapping ---")
safe_result_mapped = extractor_mapped.get_regional_ts_by_id_safe({"region_code": 2121564})
print(f"Success: {safe_result_mapped['success']}")
print(f"Entity ID resolved: {safe_result_mapped['seasonfield_id']}")

### 🧪 Test single entity processing with mapped column

In [ ]:
# Test process_single_entity_regional with a Series using the custom column name
print("--- Test: process_single_entity_regional with column_mapping ---")
row_mapped = pd.Series({"region_code": 2121564})
result_single_mapped = extractor_mapped.process_single_entity_regional(row_mapped)

print(f"Error: {result_single_mapped['error']}")
if result_single_mapped["observed_data"] is not None:
    print(f"✅ Observed data: {result_single_mapped['observed_data'].shape}")
    display(result_single_mapped["observed_data"].head(3))
if result_single_mapped["daily_avg_data"] is not None:
    print(f"✅ Daily avg data: {result_single_mapped['daily_avg_data'].shape}")

### 🧪 Test bulk parallel extraction with mapped column

In [ ]:
# Test bulk extraction with the custom 'region_code' column (mapped from 'amu_id')
print("--- Test: bulk parallel extraction with column_mapping ---")
top5_custom = amu_df_custom.head(5)
print(f"Input DataFrame columns: {top5_custom.columns.tolist()}")
print(f"Processing {len(top5_custom)} entities...\n")

results_mapped = extractor_mapped.process_entity_regional_bulk_parallel(
    entity_list=top5_custom,
    max_workers=2,
    output_path=manager.output_result_dir,
    partial_frequency=50,
    skip_export=True,  # skip export for this test
    prefix="regional_mapped_test"
)

print(f"\n--- Results summary ---")
print(f"Total: {results_mapped['total_calculations']}")
print(f"Successful: {results_mapped['successful_calculations']}")
print(f"Failed: {results_mapped['failed_calculations']}")
if not results_mapped["observed_results_df"].empty:
    print(f"Observed results shape: {results_mapped['observed_results_df'].shape}")
    display(results_mapped["observed_results_df"].head(3))

In [ ]:
# Get the clean DataFrame from the original (unmapped) extraction
observed_df = results["results_df"]  # primary frame (alias: observed_results_df)
print(observed_df.columns)

## **🌦️ Step 5: Single-entity smoke test for new weather indexes**

Runs `get_regional_ts_by_id` against a single AMU for each weather index
(`etp`, `max-wind-speed`, `p-etp`, `relative-humidity`, `snow-depth`, `solar-radiation`).
Each call reconfigures the extractor with `indicatorTypeIds=[2]` (WEATHER OBSERVED ECMWF)
and reuses the Germany Districts AMU used earlier in the notebook.

In [ ]:
# Single-entity smoke test for each new weather index.
# Reuses the original `extractor` instance — only `index` is swapped between calls.
# All weather indexes use indicatorTypeIds=[2] (WEATHER OBSERVED ECMWF).
new_indexes = [
    "etp",
    "max-wind-speed",
    "p-etp",
    "relative-humidity",
    "snow-depth",
    "solar-radiation",
]

SMOKE_AMU_ID = 2432528  # same Germany Districts AMU used earlier in the notebook
smoke_results = {}

for idx in new_indexes:
    print(f"\n=== {idx} ===")
    extractor.setup_regional_parameters(
        index=idx,
        start_date="2024-01-01",
        end_date="2024-12-31",
        fillyeargap=False,
        idblock=281,
        idpixeltype=1,
        indicatorTypeIds=[2],  # WEATHER OBSERVED ECMWF
    )
    safe = extractor.get_regional_ts_by_id_safe({"amu_id": SMOKE_AMU_ID})
    if not safe["success"]:
        print(f"  ❌ API failed: {safe['error']}")
        smoke_results[idx] = {"success": False, "error": safe["error"]}
        continue

    data = safe["data"] or {}
    n_obs = len(data.get("observedMeasures") or [])
    n_avg = len(data.get("dailyAverage") or [])
    print(f"  ✅ observedMeasures: {n_obs}, dailyAverage: {n_avg}")

    observed_df, daily_avg_df = extractor.format_regional_json(data, entity_id=SMOKE_AMU_ID)
    smoke_results[idx] = {
        "success": True,
        "observed_df": observed_df,
        "daily_avg_df": daily_avg_df,
    }
    if not observed_df.empty:
        display(observed_df.head(3))

print("\n--- Summary ---")
for idx, r in smoke_results.items():
    if r["success"]:
        print(f"  {idx}: observed={len(r['observed_df'])}, daily_avg={len(r['daily_avg_df'])}")
    else:
        print(f"  {idx}: FAILED — {r['error']}")